# Localization Updates Demo

This notebook exercises the changes on the `feature/localization-updates` branch:

1. **`bias_adjust_model_to_station` sourced from HDP** — station observations now come from the HDP catalog (~15k stations, 28 networks) instead of the old hardcoded HadISD zarr paths.
2. **Legacy airport-code compatibility** — old HadISD-style codes like `"KSAC"` still resolve, now to an HDP `station_id`.
3. **Single-network enforcement** — bias correction rejects station lists that mix HDP networks.

In [1]:
import climakitae as ck
from climakitae.new_core.data_access.data_access import DataCatalog

cd = ck.ClimateData()

INFO:climakitae.new_core.user_interface:Initializing ClimateData interface


INFO:climakitae.new_core.dataset_factory:DatasetFactory initialized with 3 validators and 12 processors


INFO:climakitae.new_core.user_interface:ClimateData initialization successful


INFO:climakitae.new_core.user_interface:✅ Ready to query!


## 1. `bias_adjust_model_to_station`: HadISD → HDP

The processor now loads station observations from the HDP catalog instead of hardcoded HadISD S3 zarr paths. Pass an HDP `station_id` (optionally `"network_id:station_id"`); all stations in one call must share a single network.

The model-side `variable_id` stays `"t2"` (WRF's native 2m temperature variable) — the processor treats `t2`/`tas` as equivalent internally, and the output is renamed to the station's HDP display name.

In [2]:
bias_corrected = (
    ck.ClimateData()
    .catalog("cadcat")
    .activity_id("WRF")
    .institution_id("UCLA")
    .variable_id("t2")
    .table_id("1hr")
    .grid_label("d03")
    .experiment_id("historical")
    .processes(
        {
            "bias_adjust_model_to_station": {
                "stations": ["ASOSAWOS_69007093217"],
                "historical_slice": (2000, 2003),  # small window for a quick demo
            },
        }
    )
    .get()
)
print(list(bias_corrected.data_vars))  # one variable per station, named by its HDP station_name
bias_corrected

INFO:climakitae.new_core.user_interface:Initializing ClimateData interface


INFO:climakitae.new_core.dataset_factory:DatasetFactory initialized with 3 validators and 12 processors


INFO:climakitae.new_core.user_interface:ClimateData initialization successful


INFO:climakitae.new_core.user_interface:✅ Ready to query!


INFO:climakitae.new_core.user_interface:Catalog set to: cadcat


...

INFO:climakitae.new_core.processors.update_attributes:UpdateAttributes applied to result; added 5 attributes


INFO:climakitae.new_core.dataset:All processing steps completed successfully


INFO:climakitae.new_core.user_interface:✅ Data retrieval successful!


['FRITZSCHE AAF']


<xarray.Dataset> Size: 14MB
Dimensions:            (sim: 5, time: 297840)
Coordinates:
  * sim                (sim) object 40B 'wrf_ucla_ec-earth3-veg_historical_r1...
  * time               (time) object 2MB 1980-09-01 00:00:00 ... 2014-08-31 2...
    Lambert_Conformal  int32 4B 1
    lakemask           float32 4B 0.0
    landmask           float32 4B 1.0
    lat                float32 4B 36.68
    lon                float32 4B -121.8
    x                  float64 8B -4.203e+06
    y                  float64 8B 1.265e+06
Data variables:
    FRITZSCHE AAF      (sim, time) float64 12MB dask.array<chunksize=(1, 297840), meta=np.ndarray>
Attributes: (12/121)
    description:                      temp at 2 m
    grid_mapping:                     Lambert_Conformal
    units:                            K
    AERCU_FCT:                        1.0
    AERCU_OPT:                        0
    AUTO_LEVELS_OPT:                  2
    ...                               ...
    bias_adjustment:                  QuantileDeltaMapping(group=Grouper(name...
    bias_adjust_model_to_station:     {'stations': ['ASOSAWOS_69007093217'], ...
    update_attributes:                Process 'update_attributes' applied to ...
    filter_unadjusted_models:         Process 'filter_unadjusted_models' appl...
    drop_leap_days:                   Leap days (February 29) have been remov...
    concat:                           Process 'concat' applied to the data. M...

## 2. Legacy airport codes still work

Existing code that references old HadISD-style airport codes (e.g. `"KSAC"`) doesn't break — these are translated to their HDP ASOSAWOS `station_id` automatically.

In [3]:
from climakitae.new_core.processors.processor_utils import (
    resolve_airport_code_to_hdp_station_id,
)

legacy_lookup = DataCatalog()["stations"]
print("KSAC ->", resolve_airport_code_to_hdp_station_id("KSAC", legacy_lookup))

KSAC -> ASOSAWOS_72483023232


## 3. Single-network enforcement

Mixing stations from two different HDP networks in one `bias_adjust_model_to_station` call is rejected — look for the warning above the printed result — rather than silently picking one network or producing inconsistent output.

In [4]:
mixed_network_result = (
    ck.ClimateData()
    .catalog("cadcat")
    .activity_id("WRF")
    .institution_id("UCLA")
    .variable_id("t2")
    .table_id("1hr")
    .grid_label("d03")
    .experiment_id("historical")
    .processes(
        {
            "bias_adjust_model_to_station": {
                # Two different HDP networks in one call -> rejected
                "stations": ["ASOSAWOS_69007093217", "SNOTEL_1000"],
            },
        }
    )
    .get()
)
print("Result:", mixed_network_result)

INFO:climakitae.new_core.user_interface:Initializing ClimateData interface


INFO:climakitae.new_core.dataset_factory:DatasetFactory initialized with 3 validators and 12 processors


INFO:climakitae.new_core.user_interface:ClimateData initialization successful


INFO:climakitae.new_core.user_interface:✅ Ready to query!


INFO:climakitae.new_core.user_interface:Catalog set to: cadcat


...

INFO:climakitae.new_core.param_validation.abc_param_validation:Checking processes ...



Processor bias_adjust_model_to_station with value {'stations': ['ASOSAWOS_69007093217', 'SNOTEL_1000']} is not valid. 
Please check the processor documentation for valid options.


INFO:climakitae.new_core.param_validation.cadcat_param_validator:Query validation result: False


ERROR:climakitae.new_core.dataset:Parameter validation failed


Result: None


## Summary

| Change | Behavior |
|---|---|
| `bias_adjust_model_to_station` | Now sources observations from HDP (28 networks, ~15k stations) instead of HadISD |
| Legacy airport codes | Still accepted, resolved to an HDP `station_id` under the hood |
| Multi-network station lists | Rejected with a clear validation warning |